In [1]:
!pip install polars

In [2]:
import gc
import numpy as np
import pandas as pd
import polars as pl
import optuna
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
import pickle

In [3]:
try:
    import pyarrow
    print("PyArrow is installed. Version:", pyarrow.__version__)
except ModuleNotFoundError:
    print("PyArrow is NOT found in this environment. You installed it in a different one.")

PyArrow is installed. Version: 23.0.1


In [4]:
target = pl.read_parquet('/kaggle/input/datasets/mashamalyshkina/kyberpolka-dataset/train_target.parquet')

In [5]:
target_columns = [col for col in target.columns if col.startswith("target")]

In [6]:
target_col_means = (
    target.select(pl.col(target_columns).mean())
    .transpose(include_header = True, column_names = ["Mean"])
    .rename({"column": "Target"})
)
print(target_col_means)

shape: (41, 2)
┌─────────────┬──────────┐
│ Target      ┆ Mean     │
│ ---         ┆ ---      │
│ str         ┆ f64      │
╞═════════════╪══════════╡
│ target_1_1  ┆ 0.010396 │
│ target_1_2  ┆ 0.003425 │
│ target_1_3  ┆ 0.023785 │
│ target_1_4  ┆ 0.023429 │
│ target_1_5  ┆ 0.001839 │
│ …           ┆ …        │
│ target_9_5  ┆ 0.006583 │
│ target_9_6  ┆ 0.223072 │
│ target_9_7  ┆ 0.077248 │
│ target_9_8  ┆ 0.010433 │
│ target_10_1 ┆ 0.315052 │
└─────────────┴──────────┘


In [7]:
ultra_rare = (
    target_col_means
    .filter(pl.col("Mean") < 0.005)
    .select("Target")
    .to_series()
    .to_list()
)
normal_targets = (
    target_col_means
    .filter(pl.col("Mean") >= 0.005)
    .select("Target")
    .to_series()
    .to_list()
)

In [8]:
train = pl.read_parquet('/kaggle/input/datasets/mashamalyshkina/kyberpolka-1300/train_combined_1300.parquet')
test = pl.read_parquet('/kaggle/input/datasets/mashamalyshkina/kyberpolka-1300/test_combined_1300.parquet')

print('Тренировочные данные:', train.shape)
print('Тестовые данные:', test.shape)

Тренировочные данные: (750000, 1301)
Тестовые данные: (250000, 1301)


In [9]:
X_train_final = train.drop('customer_id')
y_train_final = target.drop('customer_id')
X_test = test.drop('customer_id')
X_train_final.shape

(750000, 1300)

In [10]:
X_np = X_train_final.to_numpy().astype('float32')

In [11]:
y_np = (
    target
    .drop('customer_id')
    .to_numpy()
    .astype('int8')
)

In [12]:
X_test_np = X_test.to_numpy().astype('float32')

In [13]:
del X_train_final
del target
del X_test

import gc
gc.collect()

0

In [14]:
X_np.shape

(750000, 1300)

In [15]:
with open('/kaggle/input/datasets/mashamalyshkina/kyberpolka-dataset/best_params_dict.pkl', 'rb') as f:
    best_params_dict = pickle.load(f)

In [16]:
raw_preds = np.zeros((X_test_np.shape[0], len(target_columns)), dtype=np.float32)

for i, t in enumerate(target_columns):
    params = best_params_dict[t]
    model = HistGradientBoostingClassifier(**params)
    model.fit(X_np, y_np[:, i])  
    raw_preds[:, i] = model.decision_function(X_test_np)
        
    del model
    gc.collect()

In [17]:
predict_schema = [f"predict_{t.split('target_')[1]}" for t in target_columns]

hgb_submit = pl.DataFrame(
    raw_preds,
    schema=predict_schema
)

In [18]:
test_ids = pl.read_parquet(
    "/kaggle/input/datasets/mashamalyshkina/kyberpolka-dataset/test_main_features.parquet",
    columns=["customer_id"]
)

In [19]:
hgb_submit = pl.concat(
    [test_ids, hgb_submit],
    how="horizontal"
)

hgb_submit.write_parquet("hgb_optuna_1300.parquet")